# Feature Engineering

This notebook creates new features from the cleaned e-commerce dataset to support deeper analysis and modeling.  
Features are grouped by domain: Time, Financial, Discount, Customer, Product, Logistics, and Risk/Behavior.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
# Load the cleaned dataset
ecommerce_df = pd.read_csv(
    'dataset/cleaned/ecommerce_cleaned.csv',
    parse_dates=['order_date']
)

In [3]:
ecommerce_df.sample(5)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
718411,2026-01-21 09:16:08.405572,2026,1,No,Cathy Woods,Female,56,Premium,United Kingdom,Completed,Home,Appliances,44.58,1,10,40.12,19.34,48.21,Credit Card,Economy,11.81,2,United Kingdom,1,65.20,No,3.20,13,No,97.80,Mobile,Organic,Email
10925,2024-04-07 09:17:09.182042,2024,4,Yes,Krystal Wallace,Female,34,Premium,Germany,Completed,Clothing,Accessories,114.49,1,0,114.49,54.97,48.01,Bank Transfer,Express,0.91,12,Germany,2,1.40,Yes,46.60,4,Yes,67.90,Tablet,Email,Direct
410644,2025-05-06 19:41:59.463411,2025,5,No,Kevin Jackson,Male,42,Regular,France,Completed,Electronics,Tablets,341.97,1,5,324.87,183.94,56.62,Credit Card,Economy,11.96,14,France,1,10.50,No,41.60,1,Yes,2.60,Tablet,Facebook,Email
387265,2024-12-29 23:49:56.320465,2024,12,Yes,Edward Green,Male,65,Regular,United States,Completed,Health,Personal Care,150.83,5,10,678.73,283.68,41.80,Bank Transfer,Next Day,0.58,8,United States,5,61.80,Yes,4.50,13,No,59.30,Mobile,Affiliate,Email
822990,2024-07-13 09:55:18.539261,2024,7,Yes,Rhonda Hunter,Female,30,Premium,Australia,Completed,Health,Fitness,153.38,3,15,391.12,100.96,25.81,PayPal,Economy,16.05,2,Australia,3,15.90,Yes,8.20,10,No,74.80,Mobile,Email,Referral


## 1. Time Features

In [4]:
# Day of week (Mon, Tue, ...)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_month') + 1,
    'order_day_of_week',
    ecommerce_df['order_date'].dt.strftime('%a')
)

# Part of day
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_day_of_week') + 1,
    'part_day',
    pd.cut(
        ecommerce_df['order_date'].dt.hour,
        bins=[0, 6, 12, 18, 24],
        labels=['Night', 'Morning', 'Afternoon', 'Evening'],
        right=False
    )
)

## 2. Financial / Revenue Features

In [5]:
# Gross revenue before discount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'gross_revenue_usd',
    ecommerce_df.eval('quantity * unit_price_usd')
)

# Actual discount amount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('gross_revenue_usd') + 1,
    'discount_amount_usd',
    ecommerce_df.eval('gross_revenue_usd * discount_percent / 100')
)

# Shipping cost as % of revenue
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_cost_usd') + 1,
    'shipping_cost_percent',
    ecommerce_df.eval('shipping_cost_usd / revenue_usd * 100')
)

## 3. Discount Features

In [6]:
# Discount tier (adjusted to actual data range 0-25%)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_percent') + 1,
    'discount_tier',
    pd.cut(
        ecommerce_df['discount_percent'],
        bins=[-1, 0, 10, 20, 25],
        labels=['No Discount', 'Low Discount', 'Medium Discount', 'High Discount']
    )
)

# Binary flag
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_tier') + 1,
    'is_discounted',
    np.where(
        ecommerce_df.eval('discount_percent > 0'),
        "Yes",
        "No"
    )
)

## 4. Customer Features

In [7]:
# Age group
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('age') + 1,
    'age_group',
    pd.cut(
        ecommerce_df['age'],
        bins=[0, 24, 34, 44, 54, 64, 100],
        labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    )
)

# Loyalty tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('customer_loyalty_score') + 1,
    'loyalty_tier',
    pd.cut(
        ecommerce_df['customer_loyalty_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 5. Product & Order Features

In [8]:
# Price tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('unit_price_usd') + 1,
    'price_tier',
    pd.cut(
        ecommerce_df['unit_price_usd'],
        bins=[0, 50, 100, 200, np.inf],
        labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
    )
)

# Quantity / basket size
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'quantity_segment',
    pd.cut(
        ecommerce_df['quantity'],
        bins=[0, 1, 3, 5],
        labels=['Single', 'Small Basket', 'Bulk']
    )
)

# Order value segment
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('revenue_usd') + 1,
    'order_value_segment',
    pd.cut(
        ecommerce_df['revenue_usd'],
        bins=[0, 100, 300, 600, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )
)

## 6. Logistics Features

In [ ]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('delivery_days') + 1,
    'delivery_speed',
    pd.cut(
        ecommerce_df['delivery_days'],
        bins=[0, 3, 7, 15],
        labels=['Fast', 'Standard', 'Slow']
    )
)

## 7. Behavior & Risk Features

In [10]:
# Engagement level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('session_duration_min') + 1,
    'engagement_level',
    pd.cut(
        ecommerce_df['session_duration_min'],
        bins=[-1, 15, 40, 100],
        labels=['Low', 'Medium', 'High']
    )
)

# Fraud risk level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('fraud_risk_score') + 1,
    'fraud_risk_level',
    pd.cut(
        ecommerce_df['fraud_risk_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 8. Final Check & Save

In [11]:
# Quick look at the engineered dataset
ecommerce_df.sample(10)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity,quantity_segment,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,revenue_usd,order_value_segment,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_percent,delivery_days,delivery_speed,shipping_country,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
320431,2024-03-20 03:30:04.997386,2024,3,Wed,Night,No,Carlos Contreras,Male,36,35-44,Regular,Canada,Completed,Clothing,Mens Wear,119.57,Premium,1,Single,119.57,23.91,20,Medium Discount,Yes,95.66,Low,41.44,43.32,Credit Card,Express,6.86,7.17,5,Standard,Canada,1,62.20,Medium,No,36.10,Medium,19,Yes,11.50,Low,Mobile,Organic,Direct
870743,2025-08-04 10:40:06.223775,2025,8,Mon,Morning,No,William Walker,Male,54,45-54,Regular,Spain,Completed,Sports,Gym Equipment,206.39,Luxury,2,Small Basket,412.78,0.00,0,No Discount,No,412.78,High,142.66,34.56,Bank Transfer,Economy,14.17,3.43,5,Standard,Spain,3,14.90,Low,No,25.60,Medium,10,Yes,40.70,Medium,Desktop,Google Ads,Email
860813,2025-09-29 14:11:23.822774,2025,9,Mon,Afternoon,No,Megan Frederick,Female,30,25-34,Regular,Italy,Cancelled,Home,Decor,27.46,Budget,4,Bulk,109.84,0.00,0,No Discount,No,109.84,Medium,48.72,44.36,Apple Pay,Express,5.45,4.96,2,Fast,Italy,4,84.00,High,Yes,37.60,Medium,11,Yes,92.50,High,Desktop,Google Ads,Email
330851,2025-06-22 05:51:58.609054,2025,6,Sun,Night,Yes,Cynthia Cole,Female,46,45-54,Regular,Belgium,Completed,Clothing,Womens Wear,79.77,Mid-Range,5,Bulk,398.85,39.88,10,Low Discount,Yes,358.97,High,140.17,39.05,Bank Transfer,Next Day,4.56,1.27,3,Fast,Belgium,5,78.10,High,Yes,19.40,Medium,8,Yes,28.60,Low,Desktop,Facebook,Social
419828,2024-06-17 08:21:07.358078,2024,6,Mon,Morning,No,Valerie Ford,Female,75,65+,VIP,Italy,Completed,Electronics,Tablets,231.68,Luxury,2,Small Basket,463.36,23.17,5,Low Discount,Yes,440.19,High,152.93,34.74,Debit Card,Economy,6.72,1.53,4,Standard,Italy,1,85.00,High,Yes,13.10,Low,11,Yes,77.40,High,Tablet,Affiliate,Email
728893,2025-05-28 01:39:22.166595,2025,5,Wed,Night,No,Robert Hernandez,Male,47,45-54,Premium,Canada,Pending,Health,Supplements,102.11,Premium,3,Small Basket,306.33,30.63,10,Low Discount,Yes,275.70,Medium,142.50,51.69,Apple Pay,Standard,18.17,6.59,1,Fast,Canada,2,29.00,Low,Yes,17.60,Medium,9,No,42.30,Medium,Tablet,Facebook,Direct
864020,2025-04-13 01:37:47.547376,2025,4,Sun,Night,Yes,Micheal Bond,Male,20,18-24,Regular,Germany,Returned,Sports,Sports Wear,165.27,Premium,2,Small Basket,330.54,33.05,10,Low Discount,Yes,297.49,Medium,117.55,39.51,Apple Pay,Standard,16.51,5.55,12,Slow,Germany,1,72.70,High,Yes,27.40,Medium,9,Yes,25.40,Low,Tablet,Google Ads,Referral
370563,2024-06-22 06:19:21.396161,2024,6,Sat,Morning,Yes,Jeffrey Gilmore,Male,19,18-24,Regular,Germany,Pending,Electronics,Laptops,454.24,Luxury,3,Small Basket,"1,362.72",0.00,0,No Discount,No,"1,362.72",Very High,524.76,38.51,Credit Card,Economy,20.38,1.50,9,Slow,Germany,5,6.40,Low,No,2.60,Low,11,No,16.60,Low,Tablet,Email,Email
648593,2025-01-24 06:07:41.101674,2025,1,Fri,Morning,No,Bruce Green,Male,45,45-54,Regular,United States,Completed,Home,Decor,299.62,Luxury,2,Small Basket,599.24,59.92,10,Low Discount,Yes,539.32,High,155.20,28.78,Credit Card,Express,7.99,1.48,9,Slow,United States,1,86.90,High,No,23.10,Medium,5,No,19.60,Low,Desktop,Facebook,Referral
770101,2025-10-01 08:10:37.094998,2025,10,Wed,Morning,No,Jacob Martin,Male,27,25-34,Regular,Belgium,Completed,Health,Personal Care,51.31,Mid-Range,3,Small Basket,153.93,0.00,0,No Discount,No,153.93,Medium,50.01,32.49,PayPal,Express,15.94,10.36,6,Standard,Belgium,4,90.30,High,No,48.90,High,18,Yes,53.70,Medium,Mobile,Affiliate,Email


In [12]:
# Save the processed dataset
ecommerce_df.to_csv("dataset/processed/ecommerce_processed.csv", index=False)